In [1]:
# ============================================================================
# 02_model_training.ipynb
# Train three tree-ensemble classifiers (XGBoost, LightGBM, random forest) per
# domain under a performance-parity constraint: all three are tuned to a
# comparable held-out AUC so that any later difference in explanation content
# cannot be confounded with a difference in predictive quality. Each model is
# selected by randomised search with 4-fold cross-validation. Trained models
# and held-out predicted probabilities are written for the attribution stage.
# ============================================================================

# [Imports and global configuration]
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

ROOT = Path("..")
DATA_DIR = ROOT / "data"
PROC_DIR = DATA_DIR / "processed"
MODEL_DIR = DATA_DIR / "models"
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
for d in (MODEL_DIR, FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# [Plotting style: grayscale, publication settings, no captions]
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette("gray", n_colors=8))
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "sans-serif",
    "axes.grid": True, "grid.color": "0.85", "grid.linewidth": 0.5,
    "axes.edgecolor": "0.3", "axes.labelcolor": "0.15", "text.color": "0.15",
    "xtick.color": "0.3", "ytick.color": "0.3", "image.cmap": "gray",
    "savefig.bbox": "tight", "savefig.transparent": False,
})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"saved figure: {name}")

def save_table(df, name, index=False):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index, encoding="utf-8-sig")
    print(f"saved table: {name}")

# [Configuration]
with open(DATA_DIR / "feature_registry.json", encoding="utf-8") as f:
    registry = json.load(f)
DOMAINS = list(registry.keys())
MODELS = ["xgboost", "lightgbm", "random_forest"]
LINESTYLE = {"xgboost": "-", "lightgbm": "--", "random_forest": ":"}
SHADE = {"xgboost": "0.15", "lightgbm": "0.40", "random_forest": "0.60"}
PARITY_TOL = 0.02          # max allowed held-out AUC spread across the three
SEARCH_CAP = 10000         # subsample size used only for the parameter search
N_ITER = 6                 # randomised-search iterations
# Cap the rows used for the final fit so the pipeline runs on a modest machine.
# Set TRAIN_FIT_CAP = None to fit on the full training split (paper-scale run).
TRAIN_FIT_CAP = None
# Parallelism: the estimator uses the available cores; the search runs
# sequentially to avoid oversubscription on a small machine.
EST_JOBS, SEARCH_JOBS = -1, 1

def make_search(model_name):
    # Return (estimator, parameter distribution) for the randomised search.
    if model_name == "xgboost":
        est = XGBClassifier(tree_method="hist", eval_metric="logloss",
                            random_state=SEED, n_jobs=EST_JOBS)
        dist = {"n_estimators": randint(150, 400), "max_depth": randint(3, 8),
                "learning_rate": loguniform(0.02, 0.3),
                "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.6, 0.4),
                "min_child_weight": randint(1, 8)}
    elif model_name == "lightgbm":
        est = LGBMClassifier(random_state=SEED, n_jobs=EST_JOBS, verbose=-1)
        dist = {"n_estimators": randint(150, 400), "num_leaves": randint(15, 80),
                "learning_rate": loguniform(0.02, 0.3),
                "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.6, 0.4),
                "min_child_samples": randint(10, 60)}
    else:
        est = RandomForestClassifier(random_state=SEED, n_jobs=EST_JOBS)
        dist = {"n_estimators": randint(150, 250), "max_depth": randint(6, 16),
                "max_features": uniform(0.2, 0.6),
                "min_samples_leaf": randint(1, 20)}
    return est, dist

def load_split(domain, part):
    df = pd.read_parquet(PROC_DIR / f"{domain}_{part}.parquet")
    feats = registry[domain]["features"]
    return df["__case_id"].values, df[feats].values, df["target"].values.astype(int)

# [Train every model on every domain]
auc_rows = []
roc_store = {}      # (domain, model) -> (fpr, tpr)
cal_store = {}      # (domain, model) -> (prob_true, prob_pred)
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)

for domain in DOMAINS:
    cid_tr, Xtr, ytr = load_split(domain, "train")
    cid_te, Xte, yte = load_split(domain, "test")
    # Search subsample (final refit is always on the full training set).
    if len(Xtr) > SEARCH_CAP:
        rng = np.random.RandomState(SEED)
        pos = np.where(ytr == 1)[0]; neg = np.where(ytr == 0)[0]
        frac = SEARCH_CAP / len(Xtr)
        sel = np.concatenate([rng.choice(pos, int(len(pos) * frac), replace=False),
                              rng.choice(neg, int(len(neg) * frac), replace=False)])
        Xs, ys = Xtr[sel], ytr[sel]
    else:
        Xs, ys = Xtr, ytr
    # Rows used for the final fit (optionally capped for a modest machine).
    if TRAIN_FIT_CAP is not None and len(Xtr) > TRAIN_FIT_CAP:
        rng = np.random.RandomState(SEED)
        fit_idx = rng.choice(len(Xtr), TRAIN_FIT_CAP, replace=False)
        Xfit, yfit = Xtr[fit_idx], ytr[fit_idx]
    else:
        Xfit, yfit = Xtr, ytr
    # Resume: reuse cached predictions/models if a full set is already present.
    pred_fp = PROC_DIR / f"{domain}_test_predictions.parquet"
    cached = pred_fp.exists() and all(
        (MODEL_DIR / f"{domain}_{m}.joblib").exists() for m in MODELS)
    cached_pred = pd.read_parquet(pred_fp) if cached else None
    pred_cols = {"__case_id": cid_te, "target": yte}
    for model_name in MODELS:
        if cached and model_name in cached_pred.columns:
            proba = cached_pred[model_name].values
        else:
            est, dist = make_search(model_name)
            search = RandomizedSearchCV(est, dist, n_iter=N_ITER, scoring="roc_auc",
                                        cv=cv, random_state=SEED, n_jobs=SEARCH_JOBS, refit=True)
            search.fit(Xs, ys)
            best = search.best_estimator_
            best.fit(Xfit, yfit)                # final fit (capped if configured)
            proba = best.predict_proba(Xte)[:, 1]
            joblib.dump(best, MODEL_DIR / f"{domain}_{model_name}.joblib")
        auc = roc_auc_score(yte, proba)
        pred_cols[model_name] = proba
        fpr, tpr, _ = roc_curve(yte, proba)
        roc_store[(domain, model_name)] = (fpr, tpr)
        pt, pp = calibration_curve(yte, proba, n_bins=10, strategy="quantile")
        cal_store[(domain, model_name)] = (pt, pp)
        auc_rows.append({"domain": domain, "model": model_name, "test_auc": round(auc, 4)})
    pd.DataFrame(pred_cols).to_parquet(PROC_DIR / f"{domain}_test_predictions.parquet", index=False)
    spread = (max(r["test_auc"] for r in auc_rows if r["domain"] == domain)
              - min(r["test_auc"] for r in auc_rows if r["domain"] == domain))
    print(f"{domain:16s} AUC spread={spread:.4f} "
          f"{'(within parity)' if spread <= PARITY_TOL else '(ABOVE tolerance)'}")

auc_df = pd.DataFrame(auc_rows)
wide = auc_df.pivot(index="domain", columns="model", values="test_auc")
wide["auc_spread"] = wide.max(axis=1) - wide.min(axis=1)
wide["within_parity"] = wide["auc_spread"] <= PARITY_TOL
save_table(wide.reset_index(), "tab02_model_auc_parity", index=False)

# [Figure 04: ROC curves, one panel per domain, three models by line style]
fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.6))
for ax, domain in zip(axes.ravel(), DOMAINS):
    for model_name in MODELS:
        fpr, tpr = roc_store[(domain, model_name)]
        auc = float(auc_df[(auc_df.domain == domain) & (auc_df.model == model_name)]["test_auc"].iloc[0])
        ax.plot(fpr, tpr, linestyle=LINESTYLE[model_name], color=SHADE[model_name],
                lw=1.4, label=f"{model_name} ({auc:.3f})")
    ax.plot([0, 1], [0, 1], color="0.8", lw=0.8, linestyle="-")
    ax.set_title(registry[domain]["label"], fontsize=8)
    ax.set_xlabel("False positive rate", fontsize=7)
    ax.set_ylabel("True positive rate", fontsize=7)
    ax.legend(fontsize=6, frameon=False, loc="lower right")
fig.tight_layout()
save_fig(fig, "fig04_roc_curves")

# [Figure 05: AUC parity across models and domains]
fig, ax = plt.subplots(figsize=(7.6, 3.8))
x = np.arange(len(DOMAINS)); w = 0.26
for j, model_name in enumerate(MODELS):
    vals = [float(auc_df[(auc_df.domain == d) & (auc_df.model == model_name)]["test_auc"].iloc[0])
            for d in DOMAINS]
    ax.bar(x + (j - 1) * w, vals, width=w, color=SHADE[model_name],
           edgecolor="0.15", label=model_name)
ax.set_xticks(x); ax.set_xticklabels(DOMAINS, rotation=30, ha="right", fontsize=7)
ax.set_ylabel("Held-out AUC"); ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=7, frameon=False, ncol=3, loc="lower center")
save_fig(fig, "fig05_auc_parity")

# [Figure 06: calibration curves for the primary domain]
prim = next(d for d in DOMAINS if registry[d]["primary"])
fig, ax = plt.subplots(figsize=(4.4, 4.2))
ax.plot([0, 1], [0, 1], color="0.8", lw=0.8)
for model_name in MODELS:
    pt, pp = cal_store[(prim, model_name)]
    ax.plot(pp, pt, marker="o", ms=3, lw=1.3, linestyle=LINESTYLE[model_name],
            color=SHADE[model_name], label=model_name)
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed frequency")
ax.set_title(registry[prim]["label"], fontsize=8)
ax.legend(fontsize=7, frameon=False, loc="upper left")
save_fig(fig, "fig06_calibration_primary")

print("\nModel training complete:",
      f"{len(DOMAINS)} domains x {len(MODELS)} models trained.")
print(wide[[*MODELS, "auc_spread"]].round(3).to_string())


credit_personal  AUC spread=0.0178 (within parity)
credit_corporate AUC spread=0.0218 (ABOVE tolerance)
credit_taiwan    AUC spread=0.0010 (within parity)
credit_gmsc      AUC spread=0.0014 (within parity)
income_adult     AUC spread=0.0148 (within parity)
churn            AUC spread=0.0050 (within parity)
saved table: tab02_model_auc_parity
saved figure: fig04_roc_curves
saved figure: fig05_auc_parity
saved figure: fig06_calibration_primary

Model training complete: 6 domains x 3 models trained.
model             xgboost  lightgbm  random_forest  auc_spread
domain                                                        
churn               0.848     0.846          0.851       0.005
credit_corporate    0.898     0.911          0.889       0.022
credit_gmsc         0.866     0.865          0.866       0.001
credit_personal     0.948     0.946          0.930       0.018
credit_taiwan       0.782     0.781          0.782       0.001
income_adult        0.932     0.935          0.920       

saved figure: fig05_auc_parity
saved figure: fig06_calibration_primary

Model training complete: 6 domains x 3 models trained.
model             xgboost  lightgbm  random_forest  auc_spread
domain                                                        
churn               0.848     0.846          0.851       0.005
credit_corporate    0.898     0.911          0.889       0.022
credit_gmsc         0.866     0.865          0.866       0.001
credit_personal     0.948     0.946          0.930       0.018
credit_taiwan       0.782     0.781          0.782       0.001
income_adult        0.932     0.935          0.920       0.015
